## Imports

In [2]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import numpy as np 
import random 
import matplotlib.pyplot as plt 

In [3]:
torch.__version__

'2.13.0'

In [4]:
torchvision.__version__

'0.28.0'

## Setting the device

In [5]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [6]:
torch.cuda.is_available()

False

In [7]:
device

device(type='mps')

## Setting the Seed

In [8]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)

## Setting the Hyperparameters

In [9]:
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 3e-4
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_DIM = 256
NUM_HEADS = 8
DEPTH = 6
MLP_DIM = 512
DROP_RATE = 0.1

## Define Image Transformations

In [10]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

## Getting a Dataset

In [11]:
train_dataset = datasets.CIFAR10(root="data", train=True, download=True, transform=transform)

In [12]:
test_dataset = datasets.CIFAR10(root="data", train=False, download=True, transform=transform)

## Define Data Loaders

In [13]:
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Building Vision Transformer from Scratch

In [14]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels=in_channels, out_channels=embed_dim, kernel_size=patch_size, stride=patch_size)

        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1,1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 1 + num_patches, embed_dim))

    def forward(self, x: torch.Tensor):
        B = x.size(0)
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1,2)
        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_embed
        return x

In [15]:
class MLP(nn.Module):
    def __init__(self, in_features, hidden_features, drop_rate):
        super().__init__()
        self.fc1 = nn.Linear(in_features=in_features, out_features=hidden_features)
        self.fc2 = nn.Linear(in_features=hidden_features, out_features=in_features)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

In [16]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, drop_rate):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=drop_rate, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_dim, drop_rate)

    def forward(self, x):
        
        # Attention block
        residual = x
        x = self.norm1(x)
        x, _ = self.attn(x,x,x)
        x = x + residual

        # MLP
        residual = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = x + residual

        return x

In [17]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, num_classes, embed_dim, depth, num_heads, mlp_dim, drop_rate):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.encoder = nn.Sequential(
            *[TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, drop_rate) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.encoder(x)
        x = self.norm(x)
        cls_token = x[:, 0]
        return self.head(cls_token)

## Instantiate Model 

In [18]:
model = VisionTransformer(
    IMAGE_SIZE, PATCH_SIZE, CHANNELS, NUM_CLASSES, EMBED_DIM, DEPTH, NUM_HEADS, MLP_DIM, DROP_RATE
).to(device)

In [19]:
model

VisionTransformer(
  (patch_embed): PatchEmbedding(
    (proj): Conv2d(3, 256, kernel_size=(4, 4), stride=(4, 4))
  )
  (encoder): Sequential(
    (0): TransformerEncoderLayer(
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): MLP(
        (fc1): Linear(in_features=256, out_features=512, bias=True)
        (fc2): Linear(in_features=512, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (1): TransformerEncoderLayer(
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementw

## Defining a Loss Function and an Optimizer

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

## Defining a Training Loop Function 

In [21]:
def train(model, loader, optimizer, criterion):
    # Set the model to training mode
    model.train()

    total_loss, correct = 0, 0

    for x, y in loader:
        # Moving our data to target device
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        # Forward Pass
        out = model(x)

        # Calculate Loss
        loss = criterion(out, y)

        # Perform backpropagation
        loss.backward()

        # Perform Gradient Descent
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [22]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(dim=1) == y).sum().item()
    return correct / len(loader.dataset)

## Training

In [23]:
def train(model, loader, optimizer, criterion):
    # Set the model to training mode
    model.train()

    total_loss, correct = 0, 0

    for x, y in loader:
        # Moving our data to target device
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        # Forward Pass
        out = model(x)

        # Calculate Loss
        loss = criterion(out, y)

        # Perform backpropagation
        loss.backward()

        # Perform Gradient Descent
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [24]:
train_accuracies = []
test_accuracies = []

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    test_acc = evaluate(model, test_loader)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    print(f"Epoch: {epoch+1}/{EPOCHS}, Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}%, Test acc: {test_acc:.4f}%")

Epoch: 1/10, Train loss: 1.7328, Train acc: 0.3698%, Test acc: 0.4725%
Epoch: 2/10, Train loss: 1.3774, Train acc: 0.5058%, Test acc: 0.5291%
Epoch: 3/10, Train loss: 1.2350, Train acc: 0.5568%, Test acc: 0.5693%
Epoch: 4/10, Train loss: 1.1225, Train acc: 0.5994%, Test acc: 0.5870%
Epoch: 5/10, Train loss: 1.0369, Train acc: 0.6293%, Test acc: 0.6032%
Epoch: 6/10, Train loss: 0.9591, Train acc: 0.6576%, Test acc: 0.6121%
Epoch: 7/10, Train loss: 0.8778, Train acc: 0.6889%, Test acc: 0.6223%
Epoch: 8/10, Train loss: 0.8110, Train acc: 0.7130%, Test acc: 0.6266%
Epoch: 9/10, Train loss: 0.7428, Train acc: 0.7371%, Test acc: 0.6381%
Epoch: 10/10, Train loss: 0.6698, Train acc: 0.7616%, Test acc: 0.6316%
